In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_excel('/content/ApexPlanet_DataAnalytics_Dataset.xlsx')
df

,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales,city,product,Age_Group,Revenue_Per_Unit
0,ORD100002,2025-02-25,CUST5529,Customer_227,30.0,Female,Bengaluru,Rice,Grocery,7,2829.77,19808.39,Bengaluru,Rice,Adult,2829.77
1,ORD100003,2025-10-14,CUST3127,Customer_182,63.0,Male,Bengaluru,Book,Education,5,27906.16,139530.80,Bengaluru,Book,Senior,27906.16
2,ORD100004,2025-05-13,CUST8887,Customer_487,62.0,Female,Bengaluru,Book,Education,8,37491.06,299928.48,Bengaluru,Book,Senior,37491.06
3,ORD100005,2025-12-02,CUST2515,Customer_470,65.0,Female,Kolkata,Mobile,Electronics,9,28541.36,256872.24,Kolkata,Mobile,Senior,28541.36
4,ORD100006,2025-11-20,CUST4796,Customer_380,44.0,Male,Bengaluru,Rice,Grocery,10,14036.59,140365.90,Bengaluru,Rice,Middle,14036.59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,ORD100997,2025-11-07,CUST6410,Customer_301,61.0,Male,Gaya,Laptop,Electronics,9,35405.54,318649.86,Gaya,Laptop,Senior,35405.54
996,ORD100998,2025-09-03,CUST7618,Customer_154,25.0,Male,Gaya,Shoes,Fashion,5,18454.65,92273.25,Gaya,Shoes,Young,18454.65
997,ORD100999,2025-09-30,CUST9544,Customer_140,62.0,Female,Hyderabad,Laptop,Electronics,9,12971.59,116744.31,Hyderabad,Laptop,Senior,12971.59
998,ORD101000,2025-02-21,CUST9501,Customer_241,NaN,Male,Delhi,Mobile,Electronics,9,2879.01,25911.09,Delhi,Mobile,NaN,2879.01


In [ ]:
# Create a unique list of column names
cols = pd.Series(df.columns)
for dup in cols[cols.duplicated()].unique():
    cols[cols == dup] = [
        f"{dup}_{i}" if i != 0 else dup for i in range(cols[cols == dup].shape[0])
    ]
df.columns = cols

In [ ]:
import sqlite3
conn=sqlite3.connect('sales.db')

In [ ]:
df.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)

1000

In [ ]:
# 1. Force all column names to be completely lowercase and stripped of spaces
df.columns = [str(c).lower().strip() for c in df.columns]

# 2. Print out the exact columns to see what is inside
print("ALL COLUMNS:", list(df.columns))

# 3. Filter the DataFrame to keep ONLY the first occurrence of any column name
df = df.loc[:, ~df.columns.duplicated()]
print("CLEANED COLUMNS:", list(df.columns))

# 4. Attempt the SQL write
try:
    df.to_sql("sales", conn, if_exists="replace", index=False)
    print("✨ Success! Data successfully written to SQL.")
except Exception as e:
    print("❌ Failed again. Error details:")
    print(e)

ALL COLUMNS: ['order_id', 'order_date', 'customer_id', 'customer_name', 'age', 'gender', 'city', 'product', 'category', 'quantity', 'unit_price', 'total_sales', 'city', 'product', 'age_group', 'revenue_per_unit']
CLEANED COLUMNS: ['order_id', 'order_date', 'customer_id', 'customer_name', 'age', 'gender', 'city', 'product', 'category', 'quantity', 'unit_price', 'total_sales', 'age_group', 'revenue_per_unit']
✨ Success! Data successfully written to SQL.


In [ ]:
query = """
SELECT *
FROM sales
LIMIT 5;
"""

In [ ]:
#Question 1
# Total Revenue

query = """
SELECT SUM(Total_Sales) AS Total_Revenue
FROM sales;
"""

pd.read_sql(query, conn)

,Total_Revenue
0,1.393994e+08


In [ ]:
#Question 2
#Top 5 Products
query = """
SELECT Product,
       SUM(Total_Sales) AS Revenue
FROM sales
GROUP BY Product
ORDER BY Revenue DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,product,Revenue
0,Laptop,25443008.51
1,Mobile,25335573.19
2,Book,25031689.40
3,Rice,22231711.28
4,Chair,21521561.48


In [ ]:
#Question 3
#Top City
query = """
SELECT City,
       SUM(Total_Sales) AS Revenue
FROM sales
GROUP BY City
ORDER BY Revenue DESC;
"""

pd.read_sql(query, conn)

,city,Revenue
0,Patna,19285966.89
1,Kolkata,18884349.57
2,Bengaluru,18773574.32
3,Mumbai,18757050.17
4,Hyderabad,17166766.87
5,Delhi,16097079.00
6,Pune,14513175.90
7,Gaya,14380859.39
8,None,1540617.54


In [ ]:
#Question 4
#Best Category
query = """
SELECT Category,
       SUM(Total_Sales) AS Revenue
FROM sales
GROUP BY Category
ORDER BY Revenue DESC;
"""

pd.read_sql(query, conn)

,category,Revenue
0,Electronics,50778581.70
1,Education,25031689.40
2,Grocery,22231711.28
3,Furniture,21521561.48
4,Fashion,19835895.79


In [ ]:
# Question 5
# Gender Analysis
query = """
SELECT Gender,
       SUM(Total_Sales) AS Revenue
FROM sales
GROUP BY Gender;
"""

pd.read_sql(query, conn)

,gender,Revenue
0,Female,66935889.02
1,Male,72463550.63


In [ ]:
#Question 6
#Highest Spending Customer
query = """
SELECT Customer_Name,
       SUM(Total_Sales) AS Revenue
FROM sales
GROUP BY Customer_Name
ORDER BY Revenue DESC
LIMIT 1;
"""

pd.read_sql(query, conn)

,customer_name,Revenue
0,Customer_335,1684832.52


In [ ]:
# Question 7
# Average Order Value
query = """
SELECT AVG(Total_Sales) AS Avg_Order_Value
FROM sales;
"""

pd.read_sql(query, conn)

,Avg_Order_Value
0,139399.43965


In [ ]:
sql_queries = """
#Question 1
SELECT SUM(Total_Sales) AS Total_Revenue
FROM sales;

#Question 2
SELECT Product,
       SUM(Total_Sales) AS Revenue
FROM sales
GROUP BY Product;
"""